# 🏥 Medical Chatbot — Fine-tuning Qwen3.5-2B-Instruct

**Stack**: Unsloth · QLoRA 4-bit · SFTTrainer · ChatML  
**GPU target**: Kaggle T4 (15 GB) atau P100 (16 GB)

## Checklist sebelum run
1. **Dataset** — Add dataset `medical-chatbot-processed` ke notebook ini  
   (upload `train.jsonl`, `val.jsonl`, `test.jsonl` dari folder `Data/processed/`)
2. **Accelerator** — Pilih GPU: T4 x1 atau P100
3. **Internet** — Aktifkan (perlu download model dari HuggingFace)
4. **HF Token** — Tidak diperlukan (Qwen2.5 adalah model publik)


In [ ]:
%%capture
!pip install -q unsloth
!pip install -q --upgrade trl transformers accelerate peft bitsandbytes datasets
!pip install -q evaluate rouge_score
print('Dependencies installed')


In [ ]:
import os, torch
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

assert torch.cuda.is_available(), 'Enable GPU in Accelerator settings!'
gpu  = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU  : {gpu}')
print(f'VRAM : {vram:.1f} GB')
print(f'bf16 : {torch.cuda.is_bf16_supported()}')


In [ ]:
# ══════════════════════════════════════════════════
# KONFIGURASI — edit hanya cell ini
# ══════════════════════════════════════════════════

# Model
MODEL_ID    = 'unsloth/Qwen2.5-2B-Instruct-bnb-4bit'
ADAPTER_DIR = '/kaggle/working/qwen25-2b-medical-adapter'
MERGED_DIR  = '/kaggle/working/qwen25-2b-medical-merged'

# Data (dari Kaggle dataset 'medical-chatbot-processed')
DATA_DIR   = '/kaggle/input/medical-chatbot-processed'
TRAIN_FILE = f'{DATA_DIR}/train.jsonl'
VAL_FILE   = f'{DATA_DIR}/val.jsonl'

# LoRA
LORA_R      = 16
LORA_ALPHA  = 32
LORA_TARGET = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
               'gate_proj', 'up_proj', 'down_proj']

# Training (sesuai spec CLAUDE.md)
MAX_SEQ_LEN   = 2048
BATCH_SIZE    = 2      # naikkan ke 4 kalau VRAM cukup
GRAD_ACCUM    = 8      # effective batch = 2 x 8 = 16
LEARNING_RATE = 2e-4
EPOCHS        = 3
WARMUP_RATIO  = 0.05
SAVE_STEPS    = 500   # checkpoint setiap N steps (backup sesi)

# Merge full model setelah training?
DO_MERGE = True

# HuggingFace Hub (opsional)
PUSH_TO_HUB = False
HF_REPO     = 'your-username/qwen25-2b-medical'

print(f'Model  : {MODEL_ID}')
print(f'EffBatch: {BATCH_SIZE * GRAD_ACCUM}')
print(f'Steps est: {103857 // (BATCH_SIZE * GRAD_ACCUM) * EPOCHS:,} total')


In [ ]:
# ── Load Dataset ──
train_ds = load_dataset('json', data_files=TRAIN_FILE, split='train')
val_ds   = load_dataset('json', data_files=VAL_FILE,   split='train')

print(f'Train : {len(train_ds):,} samples')
print(f'Val   : {len(val_ds):,} samples')
print(f'Kolom : {train_ds.column_names}')
print('\nContoh turn pertama:')
for msg in train_ds[0]['messages']:
    print(f'  [{msg["role"]}] {msg["content"][:100]}')


In [ ]:
# ── Load Model & Tokenizer ──
print(f'Loading {MODEL_ID} ...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_ID,
    max_seq_length= MAX_SEQ_LEN,
    dtype         = None,   # auto: float16 untuk T4/P100
    load_in_4bit  = True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

params = sum(p.numel() for p in model.parameters())
print(f'Params  : {params/1e9:.2f}B')
print(f'Template: {"ada" if tokenizer.chat_template else "tidak ada - akan pakai default ChatML"}')


In [ ]:
# ── Apply Chat Template ke Dataset ──
def format_chat(sample):
    return {'text': tokenizer.apply_chat_template(
        sample['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )}

train_ds = train_ds.map(format_chat, remove_columns=['messages'], num_proc=2)
val_ds   = val_ds.map(format_chat,   remove_columns=['messages'], num_proc=2)

print('Chat template applied')
print('\nContoh teks (400 char pertama):')
print(train_ds[0]['text'][:400])


In [ ]:
# ── Konfigurasi LoRA ──
model = FastLanguageModel.get_peft_model(
    model,
    r                     = LORA_R,
    lora_alpha            = LORA_ALPHA,
    target_modules        = LORA_TARGET,
    lora_dropout          = 0,
    bias                  = 'none',
    use_gradient_checkpointing = 'unsloth',  # menghemat 30% VRAM
    random_state          = 42,
)

total    = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params     : {total/1e6:.1f}M')
print(f'Trainable params : {trainable/1e6:.1f}M ({trainable/total*100:.2f}%)')


In [ ]:
# ── Training ──
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_ds,
    eval_dataset       = val_ds,
    dataset_text_field = 'text',
    max_seq_length     = MAX_SEQ_LEN,
    dataset_num_proc   = 2,
    packing            = True,   # pack sequences pendek → +30% throughput
    args=TrainingArguments(
        output_dir                  = '/kaggle/working/checkpoints',
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE * 2,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_ratio                = WARMUP_RATIO,
        learning_rate               = LEARNING_RATE,
        optim                       = 'adamw_8bit',
        lr_scheduler_type           = 'cosine',
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 100,
        eval_strategy               = 'epoch',
        save_strategy               = 'steps',
        save_steps                  = SAVE_STEPS,
        save_total_limit            = 3,
        load_best_model_at_end      = True,
        metric_for_best_model       = 'eval_loss',
        greater_is_better           = False,
        report_to                   = 'none',
        dataloader_num_workers      = 2,
        seed                        = 42,
    ),
)

print('Training dimulai...')
result = trainer.train()
print(f'Train loss : {result.training_loss:.4f}')
print(f'Steps      : {result.global_step}')


In [ ]:
# ── Save Adapter ──
os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'Adapter saved → {ADAPTER_DIR}')

# ── Merge & Save Full Model ──
if DO_MERGE:
    os.makedirs(MERGED_DIR, exist_ok=True)
    # merged_16bit: float16, siap untuk GGUF conversion
    model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')
    print(f'Merged saved  → {MERGED_DIR}')

import subprocess
out = subprocess.run(['du', '-sh', '/kaggle/working/'], capture_output=True, text=True)
print(f'Total output  : {out.stdout.strip()}')


In [ ]:
# ── Evaluasi ROUGE-L pada test.jsonl ──
import json
from evaluate import load as load_metric

rouge = load_metric('rouge')
FastLanguageModel.for_inference(model)

SYSTEM = 'You are a helpful medical assistant. Answer questions accurately based on clinical guidelines and ICD-11 classification.'

preds, refs = [], []
test_ds = load_dataset('json', data_files=f'{DATA_DIR}/test.jsonl', split='train')

# Evaluasi pada 200 sampel pertama (full eval butuh waktu lama)
for sample in test_ds.select(range(200)):
    msgs = sample['messages']
    ref  = next(m['content'] for m in msgs if m['role'] == 'assistant')
    prompt_msgs = [m for m in msgs if m['role'] != 'assistant']

    text   = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to('cuda')

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    pred = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    preds.append(pred)
    refs.append(ref)

scores = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
print('ROUGE scores (200 test samples):')
for k, v in scores.items():
    print(f'  {k}: {v:.4f}')
print(f'\nTarget ROUGE-L > 0.30: {"PASSED" if scores["rougeL"] > 0.30 else "BELUM TERCAPAI"}')


In [ ]:
# ── Qualitative Inference Test ──
test_questions = [
    'Apa gejala utama demam berdarah dengue dan kapan harus ke rumah sakit?',
    'Bagaimana cara menangani pasien dengan nyeri dada akut?',
    'What are the first-line treatment options for Type 2 Diabetes Mellitus?',
    'Apakah antibiotik diperlukan untuk infeksi virus seperti flu biasa?',
    'Dok saya hamil 8 minggu dan mengalami mual parah, apa yang harus saya lakukan?',
]

for q in test_questions:
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user',   'content': q},
    ]
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to('cuda')

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens      = 350,
            temperature         = 0.7,
            do_sample           = True,
            top_p               = 0.9,
            repetition_penalty  = 1.1,
            pad_token_id        = tokenizer.eos_token_id,
        )
    resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'Q: {q}')
    print(f'A: {resp[:400]}')
    print('─' * 70)


In [ ]:
# ── (Opsional) Push ke HuggingFace Hub ──
# Cara aktifkan: set PUSH_TO_HUB=True di cell config,
# tambahkan secret HF_TOKEN di Kaggle: Add-ons > Secrets
if PUSH_TO_HUB:
    from huggingface_hub import login
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
    if not HF_TOKEN:
        raise ValueError('Tambahkan HF_TOKEN di Kaggle Secrets')
    login(token=HF_TOKEN)
    model.push_to_hub(HF_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
    print(f'Pushed → huggingface.co/{HF_REPO}')
else:
    print('Output tersimpan di /kaggle/working/')
    print(f'  Adapter : {ADAPTER_DIR}')
    print(f'  Merged  : {MERGED_DIR}')
